# Project 1 — Predicting California House Prices 🏡
### A from-scratch refresher on how machine learning *actually* works

**Goal of this project:** rebuild your intuition for the *complete* machine-learning workflow on simple, friendly data — so that every later project (image recognition, OCR, AI agents) sits on solid ground.

We are deliberately **not** using deep learning yet. Tabular data with a simple model lets you *see* every concept clearly, without neural-network complexity hiding the logic.

---
### The 7 universal steps of (almost) every ML project
These same 7 steps will repeat in **every** project in your journey. Learn them once here, and you'll recognise them everywhere:

1. **Frame the problem** – what are we predicting, and what *type* of problem is it?
2. **Get & understand the data (EDA)** – look before you leap.
3. **Split the data** – training set vs test set, and *why this matters so much*.
4. **Preprocess** – get the data into a form the model can use (and avoid "leakage").
5. **Train a model** – what "learning" really means.
6. **Evaluate** – measure honestly; diagnose overfitting vs underfitting.
7. **Improve** – better model, cross-validation, interpret the results.

> 🔁 **Watch for the 🔁 symbol.** It marks a *core concept that will come back in every future project.* When you see it again later, that repetition is on purpose — it's how this stuff gets drilled in.


## Step 0 — Setup

We import our tools. Think of these libraries as your kitchen equipment:
- **pandas** → the table/spreadsheet handler (our cutting board).
- **numpy** → fast math on arrays.
- **matplotlib / seaborn** → drawing charts (so we can *see* the data).
- **scikit-learn (sklearn)** → the classic machine-learning toolbox: datasets, models, metrics.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()          # nicer-looking charts
pd.set_option("display.max_columns", None)
print("Setup complete ✅")

## Step 1 — Frame the problem 🔁

Before touching any code or model, answer three questions. Doing this *every time* is a habit that separates people who understand ML from people who copy tutorials.

**Q1. What are we predicting?**
The median house value in a California district.

**Q2. What *type* of problem is this?**
We are predicting a **number** (a price like `$452,600`). Predicting a number = **Regression**.

> 🔁 **Regression vs Classification** (you'll use this distinction forever):
> - **Regression** = predict a *number*. *"How much will this house sell for?"*, *"What temperature tomorrow?"*
> - **Classification** = predict a *category*. *"Will this loan default — yes or no?"* (your loan project), *"Which emotion is this face showing?"* (your emotion project).
>
> Both are **supervised learning**: we learn from examples that already have the correct answer attached.
> 🧠 *Analogy:* supervised learning is like studying flashcards. Each card has a **question** (the inputs) and the **answer** on the back (the target). The model studies thousands of cards, then we test it on cards it has never seen.

**Q3. What are our inputs and our output?**
- **Features (X)** = the inputs / clues: median income of the area, average rooms, house age, location, etc.
- **Target (y)** = the thing we predict: the house value.

> 🧠 *Analogy:* features are the **symptoms**, the target is the **diagnosis**. The model learns which symptoms point to which diagnosis.


## Step 2 — Get & understand the data (EDA) 🔁

**EDA = Exploratory Data Analysis.** Before modelling, you *investigate* your data like a detective examining clues before naming a suspect.

Why bother? Because of the oldest rule in computing: **garbage in, garbage out.** A model can only be as good as the data you feed it. If you don't understand the data, you can't fix its problems.

This dataset is built into scikit-learn, so there's nothing to download manually.


In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame              # a pandas DataFrame: rows = districts, columns = features + target
print("Shape (rows, columns):", df.shape)
df.head()

**What each column means** (always read the data dictionary — never guess):

| Column | Meaning |
|---|---|
| `MedInc` | Median income in the district (in tens of thousands of $) |
| `HouseAge` | Median house age |
| `AveRooms` | Average rooms per household |
| `AveBedrms` | Average bedrooms per household |
| `Population` | District population |
| `AveOccup` | Average household occupancy |
| `Latitude` / `Longitude` | Location |
| `MedHouseVal` | **TARGET** — median house value (in $100,000s) |

So a `MedHouseVal` of `4.5` means **$450,000**.


In [ ]:
# .info() tells us column types and whether anything is MISSING.
# Missing values are one of the most common real-world data problems.
df.info()

In [ ]:
# .describe() gives summary statistics: min, max, mean, spread.
# Look for things that seem wrong: impossible values, huge outliers, very different scales.
df.describe()

🔎 **Notice the different scales.** `Population` is in the thousands, while `AveBedrms` is around 1. This matters a lot — we'll deal with it in Step 4. Spotting it here in EDA is exactly the point.


In [ ]:
# How is the thing we're predicting distributed?
plt.figure(figsize=(7,4))
sns.histplot(df["MedHouseVal"], bins=50)
plt.title("Distribution of house values (target)")
plt.xlabel("Median house value ($100,000s)")
plt.show()

👀 You'll see a spike at the far right (around 5.0). That's a **cap** — values were clipped at $500,000. Knowing quirks like this stops you from being confused by model errors later. *This is why we do EDA.*


In [ ]:
# Correlation: which features move together with the target?
# +1 = move together strongly, -1 = move oppositely, 0 = unrelated.
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between every pair of columns")
plt.show()

**Reading this:** `MedInc` (income) has the strongest positive correlation with house value — richer areas, pricier houses. That matches common sense, which is reassuring.

> 🧠 *Big caution that you must carry into every project:* **correlation is not causation.** Income *correlating* with price doesn't prove income *causes* price. EDA reveals relationships to investigate — not final truths.


## Step 3 — Split the data: train vs test 🔁🔁🔁
### (If you remember ONE thing from this project, make it this.)

We split our data into two piles **before doing anything else** with modelling:
- **Training set** (~80%) — the examples the model is allowed to study.
- **Test set** (~20%) — examples we *hide* and only use at the very end, as a final exam.

**Why?** Because we don't care if a model can repeat answers it has already seen. We care whether it learned the *underlying pattern* well enough to handle **new, unseen** houses. That ability is called **generalisation** — and it's the entire goal of machine learning.

> 🧠 *Analogy:* imagine a student who somehow gets the exact exam questions in advance and memorises the answers. They score 100% — but learned nothing. The **test set is a sealed envelope of fresh questions.** Only performance on *those* tells you if real learning happened.


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["MedHouseVal"])   # features (the clues)
y = df["MedHouseVal"]                   # target (the answer)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% sealed away for the final exam
    random_state=42       # a fixed seed so the split is reproducible every run
)

print("Training examples:", X_train.shape[0])
print("Test examples (hidden until the end):", X_test.shape[0])

**Why `random_state=42`?** It fixes the randomness so you (and anyone reading your GitHub) get the *exact same split* every time. Reproducibility is a professional habit — without it, your results aren't trustworthy. (42 is just a popular arbitrary number; any fixed value works.)


## Step 4 — Preprocess (and avoid data leakage) 🔁🔁

Models can't always use raw data directly. **Preprocessing** shapes the data into a usable form. Here our main job is **feature scaling**.

**The scaling problem (remember the different scales from EDA?):** `Population` ranges into the thousands while `AveBedrms` is around 1. Some models — including linear regression — secretly assume bigger numbers mean more important. So the big-number feature would unfairly dominate.

**The fix — standardisation:** rescale every feature to have mean 0 and a comparable spread, so they all start on a level playing field.

> 🧠 *Analogy:* comparing people by height-in-centimetres and weight-in-kilograms — the raw numbers aren't comparable. Scaling converts everything to the same "language" so the model weighs them fairly.


### 🔁🔁 DATA LEAKAGE — the most important subtle concept in this whole project

When we scale, we compute statistics (the mean and spread) **from the training set only**, then apply those same numbers to the test set.

**Why not use the whole dataset?** Because the test set is supposed to represent *future, unseen* data. If its values influence our preprocessing, information from the "exam" has leaked into our "study materials." Your scores will look great in the notebook and then collapse in the real world.

> 🧠 *Analogy:* **data leakage = peeking at the exam answers while studying.** You'll feel confident and then fail the real test.
>
> The rule, which you will repeat in EVERY project: **`fit` on train, `transform` on both.** Learn the recipe from the training data; apply it everywhere.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit() = LEARN the mean & spread — from TRAINING data ONLY. (No peeking at the test set.)
scaler.fit(X_train)

# transform() = APPLY that learned recipe to both sets.
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Mean of each feature AFTER scaling (training):", np.round(X_train_scaled.mean(axis=0), 2))
print("Each feature now centered ~0 and on a comparable scale ✅")

## Step 5 — Train a model 🔁
### What does "training" / "learning" actually mean?

We'll start with the simplest sensible model: **Linear Regression.**

**What it does:** it draws the best straight-line relationship between the features and the price. Concretely, it learns a *weight* for each feature:

`price ≈ (w₁ × MedInc) + (w₂ × HouseAge) + ... + b`

- Each **weight (w)** says how much that feature pushes the price up or down.
- **b** (the bias/intercept) is the baseline starting point.

**So what is "learning"?** It's the computer searching for the set of weights that make the predictions as close as possible to the real prices.

To measure "how close," we use a **loss function** — a single number for *how wrong* the model currently is. For regression we typically use **Mean Squared Error**: average of (prediction − truth)². Training = **adjust the weights to make the loss as small as possible.**

> 🧠 *Analogy:* tuning an old radio. The **loss** is the static. You turn the dial (adjust weights) to minimise the static until the station is clear. The algorithm that decides *which way to turn the dial* is called **gradient descent** — you'll meet it by name again in the neural-network projects. For now, sklearn handles it for you in one line.


In [ ]:
from sklearn.linear_model import LinearRegression

lin_model = LinearRegression()
lin_model.fit(X_train_scaled, y_train)   # <-- this single line IS the "learning"

# Inspect the learned weights — what the model decided each feature is worth:
weights = pd.Series(lin_model.coef_, index=X.columns).sort_values()
print(weights)

Read those weights like sentences. A large positive weight on `MedInc` means *"higher area income strongly pushes the predicted price up,"* which agrees with what EDA hinted. A model whose weights make real-world sense is a good sign.


## Step 6 — Evaluate honestly 🔁
### Metrics + diagnosing overfitting vs underfitting

Now we open the sealed envelope (the test set) and grade the model. For **regression** there are three metrics you'll reuse constantly:

- **MAE (Mean Absolute Error)** — the average size of the mistake, in the original units. The most human-readable: *"on average we're off by \$X."*
- **RMSE (Root Mean Squared Error)** — like MAE, but it **punishes big mistakes much more** (because it squares errors first). Use it when large errors are especially bad.
- **R² (R-squared)** — the share of the variation the model explains. **1.0 = perfect**, **0 = no better than always guessing the average price.** A quick "how good overall" score.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = lin_model.predict(X_test_scaled)

mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2   = r2_score(y_test, y_pred)

print(f"MAE : {mae:.3f}  ->  about ${mae*100000:,.0f} off on average")
print(f"RMSE: {rmse:.3f}")
print(f"R²  : {r2:.3f}  ->  explains {r2*100:.0f}% of the variation in prices")

### 🔁🔁 Overfitting vs Underfitting (you will diagnose this in every project)

Compare the score on **training** data vs **test** data. The gap tells the story:

- **Underfitting** → model does *poorly on both* train and test. It's too simple to capture the pattern.
  🧠 *A student who didn't study — fails practice and the real exam.* (This is called high **bias**.)
- **Overfitting** → model does *great on train but poorly on test*. It memorised instead of learning.
  🧠 *A student who memorised the practice answers but can't handle new questions.* (High **variance**.)
- **Good fit** → strong on both, small gap. That's the goal.

This train-vs-test gap is the single most useful diagnostic in all of ML.


In [ ]:
train_r2 = lin_model.score(X_train_scaled, y_train)
test_r2  = lin_model.score(X_test_scaled,  y_test)
print(f"Training R²: {train_r2:.3f}")
print(f"Test R²    : {test_r2:.3f}")
print(f"Gap        : {abs(train_r2 - test_r2):.3f}")
print()
print("Small gap + modest score  -> the model is consistent but a bit too SIMPLE (mild underfitting).")
print("A straight line can't capture every twist in housing data, so let's try a richer model next.")

## Step 7 — Improve: a better model + cross-validation 🔁

Linear regression assumes a straight-line world. Housing prices are full of *bends* (e.g. location effects). Let's try a **Random Forest** — a model that handles curves and interactions well and is a workhorse for tabular data.

**What a Random Forest is, simply:** a **Decision Tree** is a flowchart of yes/no questions (*"Is income > 5? Is the house coastal?"*) that narrows down to a price. A single tree tends to overfit. A **Random Forest** builds *hundreds* of slightly different trees and **averages their answers**.

> 🧠 *Analogy:* asking one friend for a restaurant tip can be hit-or-miss. Asking 100 friends and taking the consensus is far more reliable. That "wisdom of the crowd" is why forests beat single trees.

Note: tree-based models **don't need scaling** (they only compare values, never add them), so we feed them the *unscaled* data. Knowing *which* models need scaling is part of understanding *why* we scale at all.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)            # unscaled is fine for trees

rf_pred = rf_model.predict(X_test)
print(f"Random Forest  MAE : {mean_absolute_error(y_test, rf_pred):.3f}")
print(f"Random Forest  RMSE: {mean_squared_error(y_test, rf_pred) ** 0.5:.3f}")
print(f"Random Forest  R²  : {r2_score(y_test, rf_pred):.3f}")
print()
print(f"Train R²: {rf_model.score(X_train, y_train):.3f}  | Test R²: {rf_model.score(X_test, y_test):.3f}")
print("Big train>test gap? That's the forest's tendency to OVERFIT — exactly the concept from Step 6, seen again.")

### 🔁 Cross-validation — a more trustworthy grade

A single train/test split can be lucky or unlucky depending on *which* houses landed in the test set. **Cross-validation (CV)** removes that luck.

**5-fold CV:** chop the data into 5 equal parts. Train on 4, test on the 1 held out. Rotate so each part gets a turn as the test set. Average the 5 scores.

> 🧠 *Analogy:* judging a student on **five different exams** instead of one. The average is fairer and tells you how *consistent* they are (the ± spread).


In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X, y, cv=5, scoring="r2"
)
print("R² on each of the 5 folds:", np.round(cv_scores, 3))
print(f"Average R²: {cv_scores.mean():.3f}  (±{cv_scores.std():.3f})")
print("The small ± means performance is stable, not a fluke of one lucky split. That's confidence.")

### Interpret the model — which features actually mattered?

A model you can't explain is hard to trust. Random Forests can report **feature importance**: how much each feature helped the predictions.


In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(7,4))
importances.plot(kind="barh")
plt.title("Which features mattered most?")
plt.xlabel("Importance")
plt.show()
print("Median income usually dominates — which lines up with our EDA correlation finding.")
print("EDA -> model -> interpretation all telling the same story = a sign you did it right.")

## 🎓 Recap — the concepts you just drilled

You completed a full ML project and, more importantly, *understood why each step exists*:

1. **Framing** — regression vs classification; features (X) vs target (y); supervised learning = learning from labelled flashcards.
2. **EDA** — look before you model; check missing values, scales, distributions, correlations; *correlation ≠ causation*.
3. **Train/test split** — the heart of ML; we care about **generalisation** to unseen data, not memorisation.
4. **Preprocessing & leakage** — scaling levels the playing field; **fit on train, transform on both**; leakage = peeking at exam answers.
5. **Training** — "learning" = adjusting weights to minimise a **loss** (tuning the radio dial); gradient descent picks the direction.
6. **Evaluation** — MAE / RMSE / R²; diagnose **overfitting vs underfitting** by the train-vs-test gap.
7. **Improvement** — stronger model (Random Forest = wisdom of the crowd), **cross-validation** for a trustworthy grade, **feature importance** for interpretation.

### 🔁 The concepts marked with 🔁 are your foundation
Every single one returns in the next projects — only the data and models change:
- **Image Classification (Project 2):** same 7 steps, but the model becomes a neural network and "training the radio dial" becomes literal gradient descent you can watch.
- **OCR (Project 3):** same evaluation mindset, applied to reading text from images.
- **AI Agents (Project 6):** same idea of giving a system inputs and judging its outputs.

### ✍️ Try it yourself (cement the learning)
1. Change `test_size` to `0.3`. Do the scores change much? Why might that be?
2. Increase the forest's `n_estimators` to `300`. Better? Slower? Worth it?
3. Add a new feature: rooms per person = `AveRooms / AveOccup`. Does it improve R²? (This is **feature engineering** — a preview of Project 2.)

When you've run this top to bottom and tried at least one experiment, tell me and we'll move to **Project 2: teaching a neural network to recognise images** — the first real step toward your OCR goal.
